From the .json measurement of the measure_laser_power_gui.py code, compute the pressure per laser parameter.

In [ ]:
import json
import argparse
from pathlib import Path
 
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

## Cell 1: Data loading

Modify the parameters as needed.

In [ ]:
# ── CLI ─────────────────────────────────────────────────────────────────────
 
parser = argparse.ArgumentParser(description="Onda 1064 laser calibration regression.")
parser.add_argument("json_file", help="Path to calibration JSON file")
parser.add_argument("--efficiency", type=float, default=26.0,
                    help="Conversion efficiency in kPa·µJ⁻¹ (default: 26)")
parser.add_argument("--targets", type=int, nargs="+", default=[4000, 4500, 5000],
                    metavar="MV", help="Target voltages in mV (default: 4000 4500 5000)")
parser.add_argument("--sample", default=None,
                    help="Sample key to load from JSON (default: first key found)")
args = parser.parse_args()
 
CONVERSION_EFFICIENCY = args.efficiency
TARGET_VOLTAGES_MV    = args.targets
 
# ── Load JSON ───────────────────────────────────────────────────────────────
 
json_path = Path(args.json_file)
if not json_path.exists():
    raise FileNotFoundError(f"JSON file not found: {json_path}")
 
with open(json_path) as f:
    raw = json.load(f)
 
# Select sample
sample_key = args.sample or next(iter(raw))
if sample_key not in raw:
    raise KeyError(f"Sample '{sample_key}' not found. Available: {list(raw.keys())}")
sample = raw[sample_key]
print(f"Loaded sample: '{sample_key}'  ({json_path.name})\n")
 
# Parse: integer keys are rep-rate blocks; "metadata" is skipped
data = {}
for key, value in sample.items():
    if key == "metadata":
        continue
    try:
        freq = int(key)
    except ValueError:
        continue
    data[freq] = {v: float(p) for v, p in value.items()}



laser_calibration_file = '/media/aleong/Audrey-experiments1/Experiments/laser_calibration/2026-06-29_onda_laser_calibration.json'

with open(laser_calibration_file, 'r') as file:
    data = json.load(file)

CONVERSION_EFFICIENCY = 26  # kPa·µJ⁻¹
TARGET_VOLTAGES_MV = [4000, 4500, 5000]  # aka the voltages used during experiments

## Cell 2: E/pulse per voltage via forced-origin P vs f regression 

In [ ]:
freqs = np.array(sorted(data.keys()), dtype=float)
voltages = [int(v) for v in next(iter(data.values())).keys()]

e_per_pulse = {}   # mV → µJ
r2_pvf      = {}   # mV → R² of P vs f fit

for v in voltages:
    powers = np.array([data[int(f)][str(v)] for f in freqs])

    # Forced-origin OLS: slope = Σ(f·p) / Σ(f²)
    slope = np.sum(freqs * powers) / np.sum(freqs ** 2)
    e_uj  = slope * 1e6  # W/Hz → J → µJ

    fitted   = slope * freqs
    ss_res   = np.sum((powers - fitted) ** 2)
    ss_tot   = np.sum(powers ** 2)          # forced-origin R²
    r2       = 1 - ss_res / ss_tot

    e_per_pulse[v] = e_uj
    r2_pvf[v]      = r2